# Intarian Welcome Auction Strategy

This notebook models the manual auction exactly as a single-price call auction:

1. The exchange chooses the clearing price that maximizes total traded volume.
2. Ties break to the higher clearing price.
3. All bids at or above the clearing price and asks at or below it execute at the clearing price.
4. Allocation uses price priority, then time priority. Since our order is submitted last, we are last at any price level we join.

The goal is not to sweep a normal order book. The goal is to find the bid price and quantity whose induced clearing price and last-in-line allocation maximize post-buyback profit.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from html import escape
import random
from typing import Dict, Iterable, List, Tuple

try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    class _Passthrough(str):
        pass
    HTML = Markdown = _Passthrough
    def display(value):
        print(value)


@dataclass(frozen=True)
class Market:
    symbol: str
    fair_value: float
    round_trip_fee: float
    buy_orders: Dict[int, int]
    sell_orders: Dict[int, int]
    max_bid_qty: int
    min_price: int
    max_price: int
    qty_step: int = 1000


@dataclass(frozen=True)
class BidResult:
    bid_price: int
    bid_qty: int
    clearing_price: int
    traded_volume: int
    allocation: int
    profit: float


@dataclass(frozen=True)
class StressResult:
    bid_price: int
    bid_qty: int
    mean_profit: float
    fifth_percentile_profit: float
    worst_profit: float
    loss_probability: float


MARKETS = [
    Market(
        symbol="DRYLAND_FLAX",
        fair_value=30,
        round_trip_fee=0,
        buy_orders={30: 30000, 29: 5000, 28: 12000, 27: 28000},
        # User correction: max volume available at the 28 ask is 30k.
        # If the screenshot's 40k value is the one to use, change 30000 to 40000.
        sell_orders={28: 30000, 31: 20000, 32: 20000, 33: 30000},
        max_bid_qty=30000,
        min_price=24,
        max_price=34,
    ),
    Market(
        symbol="EMBER_MUSHROOM",
        fair_value=20,
        round_trip_fee=0.10,
        buy_orders={20: 43000, 19: 17000, 18: 6000, 17: 5000, 16: 10000, 15: 5000, 14: 10000, 13: 7000},
        sell_orders={12: 20000, 13: 25000, 14: 35000, 15: 6000, 16: 5000, 17: 0, 18: 10000, 19: 12000},
        max_bid_qty=43000,
        min_price=12,
        max_price=23,
    ),
]

## Auction Engine

The functions below implement the clearing-price rule and our allocation. The key subtlety is that our bid can move the clearing price, and because we are last, existing same-price orders are ahead of us.

In [2]:
def clearing_result(bids: Dict[int, int], asks: Dict[int, int], price_range: Iterable[int]) -> Tuple[int, int]:
    best_price = None
    best_volume = -1

    for price in price_range:
        demand = sum(qty for bid_price, qty in bids.items() if bid_price >= price)
        supply = sum(qty for ask_price, qty in asks.items() if ask_price <= price)
        volume = min(demand, supply)

        if volume > best_volume or (volume == best_volume and (best_price is None or price > best_price)):
            best_price = price
            best_volume = volume

    if best_price is None:
        raise ValueError("price_range must contain at least one price")
    return best_price, best_volume


def simulate_bid(market: Market, bid_price: int, bid_qty: int) -> BidResult:
    bids_with_us = dict(market.buy_orders)
    bids_with_us[bid_price] = bids_with_us.get(bid_price, 0) + bid_qty

    low = min(market.min_price, min(market.buy_orders), min(market.sell_orders), bid_price) - 1
    high = max(market.max_price, max(market.buy_orders), max(market.sell_orders), bid_price) + 1
    clearing_price, traded_volume = clearing_result(bids_with_us, market.sell_orders, range(low, high + 1))

    if bid_price < clearing_price or bid_qty <= 0:
        allocation = 0
    else:
        eligible_supply = sum(qty for ask_price, qty in market.sell_orders.items() if ask_price <= clearing_price)
        higher_priority_bids = sum(qty for price, qty in market.buy_orders.items() if price > bid_price)
        same_price_existing = market.buy_orders.get(bid_price, 0)
        allocation = max(0, min(bid_qty, eligible_supply - higher_priority_bids - same_price_existing))

    profit_per_unit = market.fair_value - clearing_price - market.round_trip_fee
    profit = allocation * profit_per_unit
    return BidResult(bid_price, bid_qty, clearing_price, traded_volume, allocation, profit)


def scan_market(market: Market) -> List[BidResult]:
    results = []
    for bid_price in range(market.min_price, market.max_price + 1):
        for bid_qty in range(0, market.max_bid_qty + market.qty_step, market.qty_step):
            results.append(simulate_bid(market, bid_price, bid_qty))
    return results


def best_result(results: List[BidResult]) -> BidResult:
    return max(results, key=lambda result: (result.profit, -result.bid_qty, -result.bid_price))


def money(value: float) -> str:
    if abs(value - round(value)) < 1e-9:
        return f"{int(round(value)):,}"
    return f"{value:,.2f}"


def qty(value: int) -> str:
    return f"{value:,}"

In [3]:
all_results = {market.symbol: scan_market(market) for market in MARKETS}
summary_rows = []
for market in MARKETS:
    best = best_result(all_results[market.symbol])
    summary_rows.append(
        f"<tr><td>{escape(market.symbol)}</td><td>{best.bid_price}</td><td>{qty(best.bid_qty)}</td>"
        f"<td>{best.clearing_price}</td><td>{qty(best.allocation)}</td><td>{money(best.profit)}</td></tr>"
    )

display(HTML(f"""
<table>
  <thead>
    <tr><th>Product</th><th>Bid price</th><th>Bid quantity</th><th>Clearing price</th><th>Allocation</th><th>Profit</th></tr>
  </thead>
  <tbody>{''.join(summary_rows)}</tbody>
</table>
"""))

Product,Bid price,Bid quantity,Clearing price,Allocation,Profit
DRYLAND_FLAX,24,0,30,0,0
EMBER_MUSHROOM,19,"40,000",18,"40,000","76,000"


## Profit Heatmaps

Each cell is the profit from submitting that single bid price and quantity. Brighter green is better; the gold outline marks the optimizer's selected order.

In [4]:
def render_heatmap(market: Market, results: List[BidResult]) -> HTML:
    by_key = {(result.bid_price, result.bid_qty): result for result in results}
    positive_profits = [result.profit for result in results if result.profit > 0]
    max_profit = max(positive_profits) if positive_profits else 1
    best = best_result(results)

    header = ''.join(
        f"<th>{qty(bid_qty)}</th>" for bid_qty in range(0, market.max_bid_qty + market.qty_step, market.qty_step)
    )
    rows = []
    for bid_price in range(market.max_price, market.min_price - 1, -1):
        cells = []
        for bid_qty in range(0, market.max_bid_qty + market.qty_step, market.qty_step):
            result = by_key[(bid_price, bid_qty)]
            intensity = 0 if result.profit <= 0 else result.profit / max_profit
            green = int(44 + 155 * intensity)
            red = int(36 + 50 * (1 - intensity))
            blue = int(56 + 20 * (1 - intensity))
            color = f"rgb({red},{green},{blue})" if result.profit > 0 else "rgb(43,43,50)"
            outline = " best" if result == best else ""
            title = (
                f"bid {bid_price} x {qty(bid_qty)}; clear {result.clearing_price}; "
                f"allocation {qty(result.allocation)}; profit {money(result.profit)}"
            )
            label = '' if bid_qty == 0 else money(result.profit)
            cells.append(
                f'<td class="cell{outline}" style="background:{color}" title="{escape(title)}">{escape(label)}</td>'
            )
        rows.append(f"<tr><th>{bid_price}</th>{''.join(cells)}</tr>")

    return HTML(f"""
    <style>
      table {{ border-collapse: collapse; font-family: Arial, sans-serif; font-size: 12px; margin: 12px 0 28px; }}
      th, td {{ border: 1px solid #d0d0d0; padding: 6px 7px; text-align: right; white-space: nowrap; }}
      th {{ background: #f1f1f1; color: #222; }}
      td.cell {{ color: #fff; min-width: 54px; }}
      td.best {{ outline: 3px solid #f4c430; outline-offset: -3px; font-weight: 700; }}
      caption {{ caption-side: bottom; text-align: left; padding-top: 8px; color: #555; }}
    </style>
    <h3>{escape(market.symbol)}</h3>
    <table>
      <caption>Rows are bid prices. Columns are bid quantities.</caption>
      <thead><tr><th>price / qty</th>{header}</tr></thead>
      <tbody>{''.join(rows)}</tbody>
    </table>
    """)


for market in MARKETS:
    display(render_heatmap(market, all_results[market.symbol]))

price / qty,0,"1,000","2,000","3,000","4,000","5,000","6,000","7,000","8,000","9,000","10,000","11,000","12,000","13,000","14,000","15,000","16,000","17,000","18,000","19,000","20,000","21,000","22,000","23,000","24,000","25,000","26,000","27,000","28,000","29,000","30,000"
34,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"-120,000"
33,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"-90,000"
32,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"-60,000"
31,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"-30,000"
30,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
29,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
28,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
27,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
26,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
25,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


price / qty,0,"1,000","2,000","3,000","4,000","5,000","6,000","7,000","8,000","9,000","10,000","11,000","12,000","13,000","14,000","15,000","16,000","17,000","18,000","19,000","20,000","21,000","22,000","23,000","24,000","25,000","26,000","27,000","28,000","29,000","30,000","31,000","32,000","33,000","34,000","35,000","36,000","37,000","38,000","39,000","40,000","41,000","42,000","43,000"
23,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","60,900","63,800","66,700","69,600","47,500","49,400","51,300","53,200","55,100","57,000","58,900","60,800","62,700","64,600","66,500","68,400","70,300","72,200","74,100","76,000","36,900","37,800","38,700"
22,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","60,900","63,800","66,700","69,600","47,500","49,400","51,300","53,200","55,100","57,000","58,900","60,800","62,700","64,600","66,500","68,400","70,300","72,200","74,100","76,000","36,900","37,800","38,700"
21,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","60,900","63,800","66,700","69,600","47,500","49,400","51,300","53,200","55,100","57,000","58,900","60,800","62,700","64,600","66,500","68,400","70,300","72,200","74,100","76,000","36,900","37,800","38,700"
20,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","60,900","63,800","66,700","69,600","47,500","49,400","51,300","53,200","55,100","57,000","58,900","60,800","62,700","64,600","66,500","68,400","70,300","72,200","74,100","76,000","36,900","37,800","38,700"
19,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","60,900","63,800","66,700","69,600","47,500","49,400","51,300","53,200","55,100","57,000","58,900","60,800","62,700","64,600","66,500","68,400","70,300","72,200","74,100","76,000","36,900","37,800","38,700"
18,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","60,900","63,800","66,700","69,600","47,500","49,400","51,300","53,200","55,100","57,000","58,900","60,800","62,700","64,600","66,500","66,500","66,500","66,500","66,500","66,500","66,500","66,500","66,500"
17,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","42,900","46,800","50,700","54,600","58,500","62,400","66,300","70,200","74,100","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000","58,000"
16,,"4,900","9,800","14,700","19,600","19,500","23,400","27,300","31,200","35,100","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000","39,000"
15,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
14,,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Robustness Check

The challenge book is fixed, so the deterministic auction optimizer is the source of truth. The stochastic section below is for robustness to human uncertainty, for example if a displayed volume is read or entered incorrectly. It perturbs every displayed volume by +/-10% and reruns the auction.

In [5]:
def top_results(results: List[BidResult], limit: int = 20) -> List[BidResult]:
    return sorted(results, key=lambda result: result.profit, reverse=True)[:limit]


def perturb_book(orders: Dict[int, int], pct: float, rng: random.Random, qty_step: int) -> Dict[int, int]:
    perturbed = {}
    for price, volume in orders.items():
        if volume == 0:
            perturbed[price] = 0
            continue
        shock = rng.uniform(-pct, pct)
        changed = max(0, int(round(volume * (1 + shock) / qty_step)) * qty_step)
        perturbed[price] = changed
    return perturbed


def stress_test_bid(market: Market, bid_price: int, bid_qty: int, simulations: int = 500, pct: float = 0.10) -> StressResult:
    rng = random.Random(f"{market.symbol}:{bid_price}:{bid_qty}:{simulations}:{pct}")
    profits = []

    for _ in range(simulations):
        stressed_market = Market(
            symbol=market.symbol,
            fair_value=market.fair_value,
            round_trip_fee=market.round_trip_fee,
            buy_orders=perturb_book(market.buy_orders, pct, rng, market.qty_step),
            sell_orders=perturb_book(market.sell_orders, pct, rng, market.qty_step),
            max_bid_qty=market.max_bid_qty,
            min_price=market.min_price,
            max_price=market.max_price,
            qty_step=market.qty_step,
        )
        profits.append(simulate_bid(stressed_market, bid_price, bid_qty).profit)

    profits.sort()
    fifth_index = max(0, int(0.05 * len(profits)) - 1)
    loss_count = sum(1 for profit in profits if profit < 0)
    return StressResult(
        bid_price=bid_price,
        bid_qty=bid_qty,
        mean_profit=sum(profits) / len(profits),
        fifth_percentile_profit=profits[fifth_index],
        worst_profit=profits[0],
        loss_probability=loss_count / len(profits),
    )


def render_stress_table(market: Market, results: List[BidResult]) -> HTML:
    candidates = [result for result in top_results(results, 20) if result.bid_qty > 0 and result.profit > 0]
    if not candidates:
        return HTML("<p>No positive deterministic bid was found, so there is no attractive candidate to stress test.</p>")

    rows = []
    for candidate in candidates[:10]:
        stress = stress_test_bid(market, candidate.bid_price, candidate.bid_qty)
        rows.append(
            f"<tr><td>{candidate.bid_price}</td><td>{qty(candidate.bid_qty)}</td>"
            f"<td>{money(candidate.profit)}</td><td>{money(stress.mean_profit)}</td>"
            f"<td>{money(stress.fifth_percentile_profit)}</td><td>{money(stress.worst_profit)}</td>"
            f"<td>{stress.loss_probability:.1%}</td></tr>"
        )

    return HTML(f"""
    <h3>{escape(market.symbol)}</h3>
    <table>
      <thead>
        <tr>
          <th>Bid price</th><th>Bid quantity</th><th>Base profit</th><th>Mean stress profit</th>
          <th>5th pct profit</th><th>Worst profit</th><th>Loss probability</th>
        </tr>
      </thead>
      <tbody>{''.join(rows)}</tbody>
    </table>
    """)


for market in MARKETS:
    display(render_stress_table(market, all_results[market.symbol]))

<p>No positive deterministic bid was found, so there is no attractive candidate to stress test.</p>

Bid price,Bid quantity,Base profit,Mean stress profit,5th pct profit,Worst profit,Loss probability
19,"40,000","76,000","55,840","36,000","36,000",0.0%
20,"40,000","76,000","56,240","36,000","36,000",0.0%
21,"40,000","76,000","59,120","36,000","36,000",0.0%
22,"40,000","76,000","58,080","36,000","36,000",0.0%
23,"40,000","76,000","56,880","36,000","36,000",0.0%
17,"19,000","74,100","63,544.60","40,600","31,900",0.0%
18,"19,000","74,100","63,536","36,100","36,100",0.0%
19,"19,000","74,100","64,068","36,100","36,100",0.0%
19,"39,000","74,100","59,982","35,100","35,100",0.0%
20,"19,000","74,100","63,840","36,100","36,100",0.0%


## Interpretation

With `DRYLAND_FLAX` ask volume at `30,000 @ 28`, the existing `30` bid already consumes the cheap supply and produces a clearing price of `30`. Because buyback is also `30`, there is no positive-profit Dryland bid under this corrected book.

For `EMBER_MUSHROOM`, the best default-grid bid is `40,000 @ 19`. It clears at `18`, allocates `40,000`, and earns `76,000` after the `0.10` round-trip fee. Bidding `43,000` looks tempting because it is the max volume, but it pushes the clearing price to `19`, which lowers profit.

This is a threshold problem: the best order is the largest order before the auction jumps to the next clearing-price regime.

## Mathematical Framework

This is not a Black-Scholes problem. Black-Scholes is for option pricing under continuous-time diffusion assumptions. Here, the mechanism is a discrete call auction with a deterministic terminal buyback.

Useful theory:

- Call auction / market microstructure: clearing price is endogenous to your submitted order.
- Marginal impact analysis: compute how much quantity you can add before the clearing price jumps.
- Priority-aware allocation: same-price orders ahead of you reduce your allocation.
- No-arbitrage against deterministic settlement: buys are attractive only when clearing price plus fees is below buyback.
- Distributionally robust optimization: if book values are uncertain, maximize performance under a set of nearby books.
- Bayesian decision theory: if a displayed volume is ambiguous, assign probability to each state and maximize posterior expected utility.
- CVaR / downside-risk control: prefer bids whose lower-tail profit remains acceptable under plausible input errors.

The best practical workflow is: exact mechanism first, sensitivity second, complexity only where uncertainty actually exists.

## Prosperity References

- [Official IMC Prosperity site](https://prosperity.imc.com/) describes Prosperity as combining algorithmic and manual trading strategies, with independent manual results.
- [IMC interview with Kevin Martin](https://www.imc.com/us/articles/discovering-imc-through-prosperity-kevins-story) notes that a sixth-place team considered complex ML but found a simple foundation worked best.
- [Frankfurt Hedgehogs Prosperity 3 writeup](https://github.com/TimoDiehm/imc-prosperity-3) is useful for tooling, visualization, robust models, backtesting, and microstructure thinking.
- [jmerle Prosperity 2 writeup](https://github.com/jmerle/imc-prosperity-2) is useful for visualization and systematic strategy development.
- [pe049395 Prosperity 2024 writeup](https://github.com/pe049395/IMC-Prosperity-2024) discusses microprice, expected utility, Monte Carlo augmentation, and Black-Scholes where option-like products appear.
- [gabsens manual challenge writeups](https://github.com/gabsens/IMC-Prosperity-2-Manual) are especially relevant for manual-market math, finite-grid optimization, probability distributions, and simulation.